# 03 - Classical ML Baselines (rematch on this project's split)

Refits the internship's own model roster (Linear, Ridge, Decision Tree, Random
Forest, Extra Trees, KNN, SVR, XGBoost, LightGBM, CatBoost) on the flat feature
table from notebook 02 — same 13 predictors the internship used, but on this
project's **chronological, burst-aware** split instead of its random 80:20 split.

Purpose: these baselines are the yardstick the CNN-LSTM (notebook 04) has to beat
on *this* split. Comparing the DL model only to the internship's original random-
split numbers would conflate two different things changing at once (split
strategy + model family); refitting classical models here isolates the model-family
effect.


In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit

from src.data import TARGET_COLS
from src.evaluate import regression_metrics, metrics_table, INTERNSHIP_BENCHMARK

pd.set_option("display.width", 120)

X_train = pd.read_csv("data/processed/X_train_flat.csv")
X_val = pd.read_csv("data/processed/X_val_flat.csv")
X_test = pd.read_csv("data/processed/X_test_flat.csv")
y_train = pd.read_csv("data/processed/y_train.csv")
y_val = pd.read_csv("data/processed/y_val.csv")
y_test = pd.read_csv("data/processed/y_test.csv")

print(X_train.shape, X_val.shape, X_test.shape)

(210, 13) (46, 13) (43, 13)


## Scaling

Fit `StandardScaler` on train only (same discipline as the internship), applied to
val/test unchanged. Train and val are concatenated for `GridSearchCV`'s internal
CV folds below (test stays untouched until final scoring) — but note the CV
strategy: `TimeSeriesSplit`, not the internship's random 5-fold, since these rows
are still chronologically ordered even within the flat table (see notebook 01/02).


In [2]:
scaler = StandardScaler().fit(X_train)
X_train_s = pd.DataFrame(scaler.transform(X_train), columns=X_train.columns)
X_val_s = pd.DataFrame(scaler.transform(X_val), columns=X_val.columns)
X_test_s = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns)

# trainval used only for GridSearchCV's internal TimeSeriesSplit folds
X_trainval_s = pd.concat([X_train_s, X_val_s], ignore_index=True)
y_trainval = pd.concat([y_train, y_val], ignore_index=True)

tscv = TimeSeriesSplit(n_splits=5)
print("Ready:", X_trainval_s.shape, "for CV; test held out at", X_test_s.shape)

Ready: (256, 13) for CV; test held out at (43, 13)


## Model roster and hyperparameter grids

Same algorithms as the internship. XGBoost/LightGBM/CatBoost are included only if
importable in this environment (guarded, so the notebook degrades gracefully rather
than failing outright if one isn't installed).


In [3]:
MODEL_GRID = {
    "Linear Regression": (LinearRegression(), {}),
    "Ridge": (Ridge(random_state=42), {"alpha": [0.1, 1.0, 10.0, 50.0]}),
    "Decision Tree": (DecisionTreeRegressor(random_state=42), {"max_depth": [3, 5, 8, None]}),
    "Random Forest": (RandomForestRegressor(random_state=42, n_estimators=200),
                       {"max_depth": [3, 5, 8, None]}),
    "Extra Trees": (ExtraTreesRegressor(random_state=42, n_estimators=200),
                     {"max_depth": [3, 5, 8, None]}),
    "KNN": (KNeighborsRegressor(), {"n_neighbors": [3, 5, 7, 11]}),
    "SVR": (SVR(), {"C": [0.1, 1.0, 10.0], "kernel": ["rbf", "linear"]}),
}

try:
    from xgboost import XGBRegressor
    MODEL_GRID["XGBoost"] = (XGBRegressor(random_state=42, verbosity=0),
                              {"max_depth": [2, 3, 4], "n_estimators": [100, 200]})
except ImportError:
    print("xgboost not installed - skipping")

try:
    from lightgbm import LGBMRegressor
    MODEL_GRID["LightGBM"] = (LGBMRegressor(random_state=42, verbose=-1),
                               {"num_leaves": [7, 15, 31], "n_estimators": [100, 200]})
except ImportError:
    print("lightgbm not installed - skipping")

try:
    from catboost import CatBoostRegressor
    MODEL_GRID["CatBoost"] = (CatBoostRegressor(random_state=42, verbose=False),
                               {"depth": [3, 4, 6], "n_estimators": [100, 200]})
except ImportError:
    print("catboost not installed - skipping")

print("Models to fit:", list(MODEL_GRID.keys()))

xgboost not installed - skipping
lightgbm not installed - skipping
catboost not installed - skipping
Models to fit: ['Linear Regression', 'Ridge', 'Decision Tree', 'Random Forest', 'Extra Trees', 'KNN', 'SVR']


## Fit, tune (GridSearchCV over TimeSeriesSplit), evaluate

One independent regressor per target (mirrors the internship's approach of two
separate single-target models) — the DL comparison in notebook 04 is where the
joint multi-task framing actually changes things.


In [4]:
results = {}
best_estimators = {}

for target in ["dbt", "wbt"]:
    for name, (estimator, grid) in MODEL_GRID.items():
        key = f"{name}"
        if grid:
            gs = GridSearchCV(estimator, grid, cv=tscv, scoring="r2", n_jobs=-1)
            gs.fit(X_trainval_s, y_trainval[target])
            fitted = gs.best_estimator_
        else:
            fitted = estimator.fit(X_trainval_s, y_trainval[target])

        best_estimators[(target, name)] = fitted

        for split_name, Xs, ys in [
            ("train", X_train_s, y_train[target]),
            ("val", X_val_s, y_val[target]),
            ("test", X_test_s, y_test[target]),
        ]:
            pred = fitted.predict(Xs)
            m = regression_metrics(ys.values, pred)
            results.setdefault(name, {}).setdefault(split_name, {})[target] = m

print("Done fitting", len(MODEL_GRID), "model types x 2 targets.")

Done fitting 7 model types x 2 targets.


In [5]:
table = metrics_table(results)
test_table = table[table.split == "test"].sort_values(["target", "R2"], ascending=[True, False])
test_table.reset_index(drop=True)

                model split target        R2      RMSE       MAE       KGE
0       Random Forest  test    dbt  0.711185  1.217756  0.947485  0.674949
1         Extra Trees  test    dbt  0.707227  1.226072  0.977585  0.672888
2               Ridge  test    dbt  0.687534  1.266637  0.968934  0.752095
3   Linear Regression  test    dbt  0.687412  1.266884  0.968475  0.761032
4                 SVR  test    dbt  0.666579  1.308421  1.000167  0.703611
5                 KNN  test    dbt  0.584900  1.459914  1.119934  0.583302
6       Decision Tree  test    dbt  0.493809  1.612158  1.125289  0.736203
7         Extra Trees  test    wbt  0.672607  0.524435  0.395698  0.786851
8                 KNN  test    wbt  0.578286  0.595203  0.400664  0.734707
9       Random Forest  test    wbt  0.434309  0.689360  0.507844  0.525927
10              Ridge  test    wbt  0.412942  0.702258  0.564670  0.711444
11  Linear Regression  test    wbt  0.410969  0.703438  0.565394  0.710972
12                SVR  te

## Rematch vs. the internship's reported numbers

Internship (random 80:20 split): DBT Ridge R2=0.6213, RMSE=1.2064, MAE=0.8792,
KGE=0.7815; WBT LightGBM R2=0.7092, RMSE=0.7372, MAE=0.5442, KGE=0.7474.

The numbers below are the same model families, same feature set, refit on the
**stricter chronological split** — expect some numbers to move; that's the point
of the harder split, not a bug.


In [6]:
for target in ["dbt", "wbt"]:
    bench = INTERNSHIP_BENCHMARK[target]
    print(f"--- {target.upper()} --- internship benchmark: {bench}")
    sub = test_table[test_table.target == target].head(3)
    print(sub[["model", "R2", "RMSE", "MAE", "KGE"]].to_string(index=False))
    print()

--- DBT --- internship benchmark: {'model': 'Ridge', 'R2': 0.6213, 'RMSE': 1.2064, 'MAE': 0.8792, 'KGE': 0.7815}
        model       R2     RMSE      MAE      KGE
Random Forest 0.711185 1.217756 0.947485 0.674949
  Extra Trees 0.707227 1.226072 0.977585 0.672888
        Ridge 0.687534 1.266637 0.968934 0.752095

--- WBT --- internship benchmark: {'model': 'LightGBM', 'R2': 0.7092, 'RMSE': 0.7372, 'MAE': 0.5442, 'KGE': 0.7474}
        model       R2     RMSE      MAE      KGE
  Extra Trees 0.672607 0.524435 0.395698 0.786851
          KNN 0.578286 0.595203 0.400664 0.734707
Random Forest 0.434309 0.689360 0.507844 0.525927



In [7]:
# Best model per target on the chronological test split, by R2 - this is the
# number the CNN-LSTM (notebook 04) needs to beat.
best_per_target = test_table.loc[test_table.groupby("target")["R2"].idxmax()]
best_per_target[["target", "model", "R2", "RMSE", "MAE", "KGE"]].reset_index(drop=True)

  target          model        R2      RMSE       MAE       KGE
0    dbt  Random Forest  0.711185  1.217756  0.947485  0.674949
1    wbt    Extra Trees  0.672607  0.524435  0.395698  0.786851

In [8]:
import pickle
from pathlib import Path

out_dir = Path("data/processed")
with open(out_dir / "baseline_results.pkl", "wb") as f:
    pickle.dump({"results": results, "table": table}, f)
print("Saved baseline_results.pkl")

Saved baseline_results.pkl
